# Extraction EDA

Run `uv run -m uni_rag_agent extract run` after inventory has populated `data/uni_rag.sqlite`, then use this notebook to inspect extraction yield, failures, chunk counts, source types, and source-location coverage.

Safety boundary: this notebook is read-only. It reads generated app data only, opens SQLite in read-only mode, enables `PRAGMA query_only=ON`, and must not mutate SQLite, `Courses`, or any course file.

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd

repo_root = Path.cwd()
sqlite_path = repo_root / "data" / "uni_rag.sqlite"
sqlite_uri = f"file:{sqlite_path.as_posix()}?mode=ro"

connection = sqlite3.connect(sqlite_uri, uri=True)
connection.execute("PRAGMA query_only=ON")

In [ ]:
latest_runs = pd.read_sql_query(
    """
    SELECT id, started_at, finished_at, status, config_json,
           files_seen, files_indexed, files_failed, error
    FROM extraction_runs
    ORDER BY id DESC
    LIMIT 20
    """,
    connection,
)
latest_runs

In [ ]:
extraction_docs = pd.read_sql_query(
    """
    SELECT extracted_documents.id AS extracted_document_id,
           files.id AS file_id,
           courses.name AS course_name,
           files.relative_path,
           files.extension,
           files.category,
           files.index_status,
           extracted_documents.extractor_name,
           extracted_documents.status AS extraction_status,
           extracted_documents.text_length,
           extracted_documents.chunk_count,
           extracted_documents.error,
           extracted_documents.extracted_at
    FROM extracted_documents
    JOIN files ON files.id = extracted_documents.file_id
    LEFT JOIN courses ON courses.id = files.course_id
    ORDER BY extracted_documents.extracted_at DESC, files.relative_path
    """,
    connection,
)
extraction_docs.head(25)

In [ ]:
coverage_by_category = pd.read_sql_query(
    """
    SELECT files.category,
           files.extension,
           files.index_status,
           COUNT(*) AS file_count
    FROM files
    WHERE files.category IN ('document', 'slides', 'notebook', 'code', 'transcript')
    GROUP BY files.category, files.extension, files.index_status
    ORDER BY files.category, files.extension, files.index_status
    """,
    connection,
)
coverage_by_category

In [ ]:
chunks = pd.read_sql_query(
    """
    SELECT chunks.id AS chunk_id,
           files.id AS file_id,
           courses.name AS course_name,
           files.relative_path,
           files.extension,
           chunks.source_type,
           chunks.location_type,
           chunks.location_value,
           chunks.token_count,
           LENGTH(chunks.text) AS text_chars
    FROM chunks
    JOIN files ON files.id = chunks.file_id
    LEFT JOIN courses ON courses.id = files.course_id
    ORDER BY files.relative_path, chunks.chunk_index
    """,
    connection,
)
chunks.head(25)

In [ ]:
chunk_source_summary = (
    chunks.groupby(["source_type", "location_type"], dropna=False)
    .agg(chunk_count=("chunk_id", "count"), median_tokens=("token_count", "median"), max_tokens=("token_count", "max"))
    .reset_index()
    .sort_values(["source_type", "location_type"])
)
chunk_source_summary

In [ ]:
failed_extractions = extraction_docs.loc[extraction_docs["extraction_status"] == "failed"]
failed_extractions[["course_name", "relative_path", "extension", "extractor_name", "error", "extracted_at"]].head(50)